# FER2013 Balanced CNN (No L2, Mild Augmentation)
Target validation accuracy: ~63–66% without transfer learning


In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.utils.class_weight import compute_class_weight
print('TensorFlow:', tf.__version__)


2026-03-04 14:46:19.413564: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772635579.625435      56 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772635579.686145      56 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772635580.183858      56 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772635580.183900      56 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772635580.183903      56 computation_placer.cc:177] computation placer alr

TensorFlow: 2.19.0


In [2]:
DATASET_PATH = '/kaggle/input/datasets/deadskull7/fer2013/fer2013.csv'
MODEL_SAVE_PATH = '/kaggle/working/fer_cnn_final.keras'

IMG_SIZE = 48
BATCH_SIZE = 64
EPOCHS = 50

df = pd.read_csv(DATASET_PATH)
train_df = df[df['Usage']=='Training']
val_df = df[df['Usage']=='PublicTest']

print('Train:',len(train_df),'Val:',len(val_df))


Train: 28709 Val: 3589


In [3]:
def preprocess(pixel_string,label):
 pixels=tf.strings.split(pixel_string)
 pixels=tf.strings.to_number(pixels,tf.float32)
 image=tf.reshape(pixels,[48,48,1])
 image=image/255.0
 label=tf.one_hot(label,7)
 return image,label

train_ds=tf.data.Dataset.from_tensor_slices((train_df['pixels'].tolist(),train_df['emotion'].tolist()))
train_ds=train_ds.map(preprocess,num_parallel_calls=2).shuffle(1000).batch(BATCH_SIZE).prefetch(1)

val_ds=tf.data.Dataset.from_tensor_slices((val_df['pixels'].tolist(),val_df['emotion'].tolist()))
val_ds=val_ds.map(preprocess,num_parallel_calls=2).batch(BATCH_SIZE).prefetch(1)


I0000 00:00:1772635610.456329      56 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1772635610.462266      56 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [4]:
class_weights=compute_class_weight(class_weight='balanced',classes=np.unique(train_df['emotion']),y=train_df['emotion'])
class_weights=dict(enumerate(class_weights))
print(class_weights)


{0: np.float64(1.0266046844269623), 1: np.float64(9.406618610747051), 2: np.float64(1.0010460615781582), 3: np.float64(0.5684387684387684), 4: np.float64(0.8491274770777877), 5: np.float64(1.293372978330405), 6: np.float64(0.8260394187886635)}


In [5]:
data_aug=tf.keras.Sequential([
 layers.RandomFlip('horizontal'),
 layers.RandomRotation(0.05)
])

model=tf.keras.Sequential([
 layers.Input(shape=(48,48,1)),
 data_aug,

 layers.Conv2D(64,(3,3),activation='relu'),
 layers.BatchNormalization(),
 layers.Conv2D(64,(3,3),activation='relu'),
 layers.BatchNormalization(),
 layers.MaxPooling2D(),
 layers.Dropout(0.25),

 layers.Conv2D(128,(3,3),activation='relu'),
 layers.BatchNormalization(),
 layers.Conv2D(128,(3,3),activation='relu'),
 layers.BatchNormalization(),
 layers.MaxPooling2D(),
 layers.Dropout(0.25),

 layers.Conv2D(256,(3,3),activation='relu'),
 layers.BatchNormalization(),
 layers.MaxPooling2D(),
 layers.Dropout(0.3),

 layers.Flatten(),
 layers.Dense(512,activation='relu'),
 layers.BatchNormalization(),
 layers.Dropout(0.5),

 layers.Dense(7,activation='softmax')
])

model.compile(optimizer=tf.keras.optimizers.Adam(5e-4),loss='categorical_crossentropy',metrics=['accuracy'])
model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 48, 48, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 46, 46, 64)     │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 46, 46, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 44, 44, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 44, 44, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 22, 22, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 22, 22, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 20, 20, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 20, 20, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 18, 18, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 18, 18, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 9, 9, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 9, 9, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 7, 7, 256)      │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 7, 7, 256)      │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 3, 3, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 3, 3, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2304)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         3,59

 Total params: 1,742,535 (6.65 MB)

 Trainable params: 1,740,231 (6.64 MB)

 Non-trainable params: 2,304 (9.00 KB)

In [7]:
history=model.fit(
 train_ds,
 validation_data=val_ds,
 epochs=EPOCHS,
 class_weight=class_weights,
 callbacks=[tf.keras.callbacks.EarlyStopping(patience=8,restore_best_weights=True)]
)

model.save(MODEL_SAVE_PATH)
print('Saved:',MODEL_SAVE_PATH)


Epoch 1/50
449/449 ━━━━━━━━━━━━━━━━━━━━ 20s 44ms/step - accuracy: 0.6142 - loss: 0.9394 - val_accuracy: 0.5974 - val_loss: 1.0771
Epoch 2/50
449/449 ━━━━━━━━━━━━━━━━━━━━ 20s 45ms/step - accuracy: 0.6155 - loss: 0.9418 - val_accuracy: 0.6060 - val_loss: 1.0569
Epoch 3/50
449/449 ━━━━━━━━━━━━━━━━━━━━ 20s 45ms/step - accuracy: 0.6203 - loss: 0.9336 - val_accuracy: 0.6071 - val_loss: 1.0775
Epoch 4/50
449/449 ━━━━━━━━━━━━━━━━━━━━ 21s 45ms/step - accuracy: 0.6144 - loss: 0.9295 - val_accuracy: 0.6032 - val_loss: 1.0876
Epoch 5/50
449/449 ━━━━━━━━━━━━━━━━━━━━ 20s 45ms/step - accuracy: 0.6184 - loss: 0.9359 - val_accuracy: 0.6096 - val_loss: 1.0759
Epoch 6/50
449/449 ━━━━━━━━━━━━━━━━━━━━ 20s 45ms/step - accuracy: 0.6210 - loss: 0.9246 - val_accuracy: 0.6158 - val_loss: 1.0698
Epoch 7/50
449/449 ━━━━━━━━━━━━━━━━━━━━ 20s 45ms/step - accuracy: 0.6254 - loss: 0.9052 - val_accuracy: 0.6219 - val_loss: 1.0401
Epoch 8/50
449/449 ━━━━━━━━━━━━━━━━━━━━ 20s 45ms/step - accuracy: 0.6290 - loss: 0.9002 - 